# Part 2: centrality vectors, a bigger dataset, and multi-domain proteins

This notebook follows the three to-dos from the meeting:

1. **Centrality vectors.** Describe each frame by one centrality value per residue (instead of one value per residue pair), then apply the same dimensionality reduction (PCA, t-SNE) as before.
2. **A bigger, more varied dataset.** mdCATH: thousands of protein domains, each simulated at five temperatures (320–450 K). At high temperatures, proteins partly or fully unfold, so the dynamics vary much more than in ATLAS.
3. **Proteins with several domains** (1k5n_A and 16pk_A have two). Project the whole protein, each domain on its own, and the contacts between domains, and compare.

The notebook is self-contained: it reloads the data and redefines the helper functions from the first notebook.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import mdtraj as md
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Extra packages for this notebook:  pip install igraph h5py huggingface_hub
import igraph as ig

## Settings

Same choices as in the first notebook: every 5th frame, contacts below 8 Å, and windows of 20 frames (1 ns) with 0.8 overlap.
`1k5n_A` must be downloaded from ATLAS like the others (10k frames, only protein) and placed in a folder `1k5n_A_protein/`.

In [ ]:
PROTEINS = ["1a62_A", "16pk_A", "1k5n_A"]

STRIDE = 5                  # every 5th frame -> ~2000 frames over 100 ns
CONTACT_THRESHOLD = 8.0     # Å
WIDTH, OVERLAP = 20, 0.8    # sliding window: 20 frames (1 ns), 80% overlap
RANDOM_STATE = 42


def file_paths(protein):
    folder = f"{protein}_protein"
    return f"{folder}/{protein}.pdb", f"{folder}/{protein}_prod_R1_fit.xtc"

## Helper functions

The same helpers as in the first notebook, collected in one cell:

- `load_ca`: load a trajectory, keep the CA atoms, coordinates in Å.
- `contact_table`: one row per frame, one column per residue pair (True = contact).
- `window_mean`: average any per-frame array over sliding windows.
- `plot_projection`: scatter plot of a 2D projection, with lines between consecutive points.

In [ ]:
def load_ca(protein, stride=STRIDE):
    pdb, xtc = file_paths(protein)
    traj = md.load(xtc, top=pdb, stride=stride)
    traj = traj.atom_slice(traj.topology.select("name CA"))
    return {"traj": traj, "coords": traj.xyz * 10, "time_ns": traj.time / 1000}


def contact_table(coords, threshold=CONTACT_THRESHOLD):
    """(frames, residue pairs) boolean table; column order = np.triu_indices(N, k=1)."""
    return np.array([pdist(frame) < threshold for frame in coords])


def step_for(width, overlap):
    return max(1, int(round(width * (1 - overlap))))


STEP = step_for(WIDTH, OVERLAP)


def window_mean(values, width=WIDTH, step=STEP):
    starts = np.arange(0, len(values) - width + 1, step)
    return np.array([values[s:s + width].mean(axis=0) for s in starts])


def plot_projection(ax, Y, colour, title, colour_label, connect=True, cmap="viridis"):
    if connect:
        ax.plot(Y[:, 0], Y[:, 1], lw=0.3, c="gray", alpha=0.6, zorder=1)
    sc = ax.scatter(Y[:, 0], Y[:, 1], c=colour, s=4, zorder=2, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(sc, ax=ax, label=colour_label)


def varying_columns(X):
    """Columns whose value changes at least once (constant columns carry no information)."""
    if X.dtype == bool:
        return X.any(axis=0) & ~X.all(axis=0)
    return X.std(axis=0) > 0


def changing_contacts(coords, threshold=CONTACT_THRESHOLD):
    """Contact table with only the contacts that change, as float (keeps memory low for big proteins)."""
    C = contact_table(coords, threshold)
    return C[:, varying_columns(C)].astype(np.float32)

In [ ]:
data = {}
for p in PROTEINS:
    if not os.path.exists(file_paths(p)[1]):
        print(f"{p}: files not found, skipped")
        continue
    data[p] = load_ca(p)
    M, N, _ = data[p]["coords"].shape
    print(f"{p}: {M} frames, {N} residues")

## 1. Centrality vectors

### The idea

So far, each frame was described by **one value per residue pair** (contact or no contact): 8,385 columns for 1a62_A.
Here, each frame is described by **one value per residue**: how central that residue is in the frame's contact network.
That gives one column per residue (130 for 1a62_A), so frame *t* becomes the vector

  **v**(t) = (centrality of residue 0, centrality of residue 1, …, centrality of residue N−1).

Then the same steps as before: windows, PCA and t-SNE, time vs. PC1.

### Four centralities

| Centrality | What it measures for a residue | In protein terms |
| --- | --- | --- |
| **Degree** | Number of contacts | How buried/packed the residue is |
| **Closeness** | 1 / average shortest-path length to all other residues | How close it is to the "middle" of the network |
| **Betweenness** | How many shortest paths between other residues pass through it | Whether it is a bridge, e.g. a hinge between domains |
| **Eigenvector** | High if its neighbours are themselves central | Whether it sits in a densely connected core |

### Why try this

- **Fewer, more interpretable features.** One number per residue, so a PCA axis can be read back as "these residues change".
- **Different notion of similarity.** Two frames are similar if every residue plays a similar role, even if the exact partner residues differ slightly.
- **What is lost:** which partner a residue touches. Degree, for example, is just the sum of a residue's contacts: residue 103 losing contact with 106 but gaining one with 38 keeps its degree the same.

In [ ]:
def frame_centralities(frame_coords, threshold=CONTACT_THRESHOLD):
    """Degree, closeness, betweenness and eigenvector centrality of every residue in one frame."""
    N = len(frame_coords)
    A = squareform(pdist(frame_coords) < threshold)          # N x N adjacency, no self-loops
    i, j = np.nonzero(np.triu(A, k=1))
    g = ig.Graph(n=N, edges=list(zip(i.tolist(), j.tolist())))
    return {
        "degree": A.sum(axis=1).astype(float),
        "closeness": np.nan_to_num(np.array(g.closeness(), dtype=float)),
        "betweenness": np.array(g.betweenness(), dtype=float),
        "eigenvector": np.array(g.eigenvector_centrality(), dtype=float),
    }


CENTRALITIES = ["degree", "closeness", "betweenness", "eigenvector"]


def centrality_vectors(coords, threshold=CONTACT_THRESHOLD):
    """dict: centrality name -> (frames, residues) array."""
    per_frame = [frame_centralities(f, threshold) for f in coords]
    return {c: np.array([fc[c] for fc in per_frame]) for c in CENTRALITIES}


cent = {}
for p in data:
    cent[p] = centrality_vectors(data[p]["coords"])
    print(f"{p}: centrality vectors, shape {cent[p]['degree'].shape} (frames, residues)")

### Projections: edge vector vs. centrality vectors

Rows: the edge vector from the first notebook (baseline), then the four centralities.
Columns: PCA, t-SNE, and time vs. PC1. All use the same windows (1 ns, 0.8 overlap).

**Scaling.** Within one centrality, all columns have the same unit, so no normalization is needed (as for the edge vector).
Set `SCALE = True` to z-score each residue's column instead; residues whose centrality barely changes then get as much weight as the others.

**What to look for:** does a centrality show the same states and transitions as the edge vector (for 1a62_A: the switch at ~77 ns)? Clearer, or weaker?

In [ ]:
PROTEIN = "1a62_A"
SCALE = False


def project(X, scale=SCALE, tsne=True):
    X = X[:, varying_columns(X)]
    if scale:
        X = StandardScaler().fit_transform(X)
    pca = PCA(n_components=2).fit(X)
    out = {"pca": pca.transform(X), "var": pca.explained_variance_ratio_, "components": pca.components_}
    if tsne:
        out["tsne"] = TSNE(n_components=2, perplexity=30, init="pca",
                           random_state=RANDOM_STATE).fit_transform(X)
    return out


def representations(p):
    """Windowed feature tables for one protein: edge vector + the four centralities."""
    reps = {"edge vector": window_mean(changing_contacts(data[p]["coords"]))}
    for c in CENTRALITIES:
        reps[c] = window_mean(cent[p][c])
    return reps


reps = representations(PROTEIN)
tW = window_mean(data[PROTEIN]["time_ns"])
proj = {name: project(X) for name, X in reps.items()}

fig, axes = plt.subplots(len(reps), 3, figsize=(17, 4.2 * len(reps)))
for row, (name, P) in zip(axes, proj.items()):
    plot_projection(row[0], P["pca"], tW, f"{name}: PCA ({P['var'].sum():.0%} in 2D)", "Time (ns)")
    plot_projection(row[1], P["tsne"], tW, f"{name}: t-SNE", "Time (ns)")
    row[2].plot(tW, P["pca"][:, 0], lw=0.8)
    row[2].set_title(f"{name}: PC1 over time", fontsize=10)
    row[2].set_xlabel("Time (ns)"); row[2].set_ylabel("PC1")
fig.suptitle(PROTEIN, y=1.0)
plt.tight_layout()
plt.show()

### Time vs. PC1 for every protein and representation

PC1 is scaled to mean 0 and standard deviation 1 per line, so the shapes can be compared (the sign of PC1 is arbitrary, so a line may be upside down).
Matching jumps across representations mean they detect the same transition.

In [ ]:
fig, axes = plt.subplots(len(data), 1, figsize=(13, 3.3 * len(data)), squeeze=False)
for ax, p in zip(axes[:, 0], data):
    t = window_mean(data[p]["time_ns"])
    for name, X in representations(p).items():
        pc1 = project(X, tsne=False)["pca"][:, 0]
        ax.plot(t, (pc1 - pc1.mean()) / pc1.std(), lw=0.8, label=name)
    ax.set_title(f"{p}: PC1 over time (standardized)", fontsize=10)
    ax.set_ylabel("PC1 (z)")
    ax.legend(fontsize=8, ncol=5, loc="upper left")
axes[-1, 0].set_xlabel("Time (ns)")
plt.tight_layout()
plt.show()

### Which residues change? Centrality over time

The advantage of one value per residue: you can draw the whole feature table as an image.
Each row is a residue, each column a window in time, colour = centrality. Horizontal bands that change colour at some time point are residues whose role changes.

Below each heatmap, the **PC1 loadings**: how strongly each residue's centrality drives PC1. Tall bars (positive or negative) mark the residues that separate the states along PC1.

In [ ]:
SHOW = ["degree", "betweenness"]
TOP = 10

p = PROTEIN
t = window_mean(data[p]["time_ns"])
fig, axes = plt.subplots(2, len(SHOW), figsize=(8 * len(SHOW), 9),
                         gridspec_kw={"height_ratios": [3, 1]}, squeeze=False)
for col, c in enumerate(SHOW):
    X = window_mean(cent[p][c])
    N = X.shape[1]
    im = axes[0, col].imshow(X.T, aspect="auto", origin="lower", cmap="magma",
                             extent=[t[0], t[-1], -0.5, N - 0.5], interpolation="nearest")
    axes[0, col].set_title(f"{p}: {c} centrality per residue")
    axes[0, col].set_xlabel("Time (ns)"); axes[0, col].set_ylabel("Residue index")
    plt.colorbar(im, ax=axes[0, col], label=c)

    loadings = np.zeros(N)
    keep = varying_columns(X)
    loadings[keep] = PCA(n_components=1).fit(X[:, keep]).components_[0]
    axes[1, col].bar(np.arange(N), loadings, width=1.0)
    axes[1, col].set_xlim(-0.5, N - 0.5)
    axes[1, col].set_xlabel("Residue index"); axes[1, col].set_ylabel("PC1 loading")
    top = np.argsort(-np.abs(loadings))[:TOP]
    print(f"{c}: residues with the largest PC1 loadings:",
          ", ".join(f"{r} ({loadings[r]:+.2f})" for r in top))
plt.tight_layout()
plt.show()

**Reading tips**

- For 1a62_A, check whether residues around 38–39 and 100–111 (the transition found earlier) appear among the top loadings.
- Betweenness picks out "bridge" residues. In a multi-domain protein, the residues connecting the domains should have high betweenness; a change in their betweenness points to a hinge motion (see section 3).
- If a centrality projection looks like the edge-vector projection, the transition is visible at residue level. If it is weaker, the change is mainly about *which* residues touch, not how many.

## 2. A bigger and more varied dataset: mdCATH

### Why mdCATH

ATLAS simulations are short (100 ns) and run at room temperature, starting from the folded structure, so the proteins mostly fluctuate around one shape.
**mdCATH** ([Mirarchi et al. 2024](https://arxiv.org/abs/2407.14794)) has:

- 5,398 protein domains (50–500 residues),
- each simulated at **five temperatures: 320, 348, 379, 413 and 450 K**, with five replicas per temperature,
- mostly **500 ns** per trajectory, one frame per ns.

At 413 K and especially 450 K many domains partly or fully unfold, so one domain gives you trajectories ranging from "stable" to "unfolding". That is the variation ATLAS lacks.

### How the data is stored

One HDF5 file per domain, downloaded from Hugging Face (about 0.6 GB per domain on average, covering all 25 trajectories). Inside:
`<domain>/pdbProteinAtoms` (the structure), and `<domain>/<temperature>/<replica>/coords` with shape (frames, atoms, 3) in Å.

### What the cells below do

1. Download one domain (or reuse a downloaded file).
2. Load one replica at several temperatures and keep the CA atoms.
3. Put **all temperatures into one shared projection**, so you can see whether high-temperature runs move away from the folded region, and whether different temperatures visit the same states.

The default domain, `5sicI00`, is the example in the mdCATH paper that stays folded at lower temperatures and unfolds at 450 K. Any other domain ID from the dataset works the same way.

In [ ]:
import h5py
from huggingface_hub import hf_hub_download

MDCATH_DOMAIN = "5sicI00"
TEMPERATURES = [320, 379, 413, 450]   # choose from 320, 348, 379, 413, 450
REPLICA = 0                           # 0-4
MDCATH_DIR = "mdcath"


def mdcath_file(domain):
    local = os.path.join(MDCATH_DIR, "data", f"mdcath_dataset_{domain}.h5")
    if os.path.exists(local):
        return local
    return hf_hub_download(repo_id="compsciencelab/mdCATH", filename=f"mdcath_dataset_{domain}.h5",
                           subfolder="data", local_dir=MDCATH_DIR, repo_type="dataset")


def load_mdcath(domain, temperatures, replica=0):
    """CA coordinates (Å) per temperature: dict T -> {"coords", "time_ns"}; plus the CA topology."""
    path = mdcath_file(domain)
    with h5py.File(path, "r") as f:
        g = f[domain]
        pdb_text = g["pdbProteinAtoms"][()]
        pdb_text = pdb_text.decode() if isinstance(pdb_text, bytes) else str(pdb_text)
        tmp = os.path.join(MDCATH_DIR, f"{domain}_protein.pdb")
        with open(tmp, "w") as out:
            out.write(pdb_text)
        top = md.load_pdb(tmp).topology
        ca = top.select("name CA")
        runs = {}
        for T in temperatures:
            coords = g[str(T)][str(replica)]["coords"][:]          # (frames, atoms, 3), Å
            if coords.shape[1] != top.n_atoms:
                raise ValueError(f"{T} K: {coords.shape[1]} atoms in coords, {top.n_atoms} in the structure")
            runs[T] = {"coords": coords[:, ca], "time_ns": np.arange(len(coords), dtype=float)}  # 1 frame per ns
    return runs, top.subset(ca)


runs, mdcath_top = load_mdcath(MDCATH_DOMAIN, TEMPERATURES, REPLICA)
for T, r in runs.items():
    print(f"{MDCATH_DOMAIN} {T} K: {r['coords'].shape[0]} frames, {r['coords'].shape[1]} residues")

### Sanity check: RMSD and radius of gyration per temperature

Before projecting, check that the temperatures really differ. RMSD to the first frame shows how far the structure moves; the radius of gyration shows how compact it is (it grows when the protein unfolds).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for T, r in runs.items():
    c = r["coords"]
    rmsd = np.sqrt(((c - c[0]) ** 2).sum(-1).mean(-1))                       # no alignment, as in notebook 1
    rg = np.sqrt(((c - c.mean(axis=1, keepdims=True)) ** 2).sum(-1).mean(-1))
    axes[0].plot(r["time_ns"], rmsd, lw=0.7, label=f"{T} K")
    axes[1].plot(r["time_ns"], rg, lw=0.7, label=f"{T} K")
axes[0].set_ylabel("RMSD to frame 0 (Å, unaligned)"); axes[1].set_ylabel("Radius of gyration (Å)")
for ax in axes:
    ax.set_xlabel("Time (ns)"); ax.legend()
plt.tight_layout()
plt.show()

### One shared projection for all temperatures

All temperatures are stacked into one table and projected together, so positions are comparable across runs.
Left: coloured by temperature. Middle: coloured by time within each run. Right: PC1 over time per temperature.

Note that mdCATH coordinates are **not pre-aligned** like the ATLAS `_fit` files; contact-based features don't need alignment, which is one more reason to use them here.
With 1 frame per ns, a window of 5 frames covers 5 ns.

In [ ]:
MDCATH_WIDTH, MDCATH_OVERLAP = 5, 0.8
mstep = step_for(MDCATH_WIDTH, MDCATH_OVERLAP)
REPRESENTATION = "edge vector"   # or one of CENTRALITIES, e.g. "betweenness"


def features(coords, representation):
    # Full contact table here: constant columns are dropped only after stacking all temperatures,
    # so every temperature keeps the same columns.
    if representation == "edge vector":
        return contact_table(coords)
    return centrality_vectors(coords)[representation]


blocks, temps, times = [], [], []
for T, r in runs.items():
    W = window_mean(features(r["coords"], REPRESENTATION), MDCATH_WIDTH, mstep)
    blocks.append(W)
    temps.append(np.full(len(W), T))
    times.append(window_mean(r["time_ns"], MDCATH_WIDTH, mstep))
X_all = np.vstack(blocks)
temps, times = np.concatenate(temps), np.concatenate(times)
X_all = X_all[:, varying_columns(X_all)]
Y = PCA(n_components=2).fit_transform(X_all)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for T in runs:
    sel = temps == T
    axes[0].plot(Y[sel, 0], Y[sel, 1], lw=0.4, alpha=0.7)
    axes[0].scatter(Y[sel, 0], Y[sel, 1], s=4, label=f"{T} K")
axes[0].legend(); axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].set_title(f"{MDCATH_DOMAIN}: {REPRESENTATION} PCA, coloured by temperature")
plot_projection(axes[1], Y, times, f"{MDCATH_DOMAIN}: coloured by time within each run", "Time (ns)", connect=False)
for T in runs:
    sel = temps == T
    axes[2].plot(times[sel], Y[sel, 0], lw=0.8, label=f"{T} K")
axes[2].set_xlabel("Time (ns)"); axes[2].set_ylabel("PC1"); axes[2].legend()
axes[2].set_title("PC1 over time per temperature")
plt.tight_layout()
plt.show()

**What to look for**

- Low temperatures should stay in one region (the folded state). High temperatures should leave it, ideally along a path that shows *how* the protein unfolds.
- If 413 K and 450 K follow the same path but 450 K goes further, they unfold the same way; separate paths mean different unfolding routes.
- To compare replicas instead of temperatures, load one temperature with several values of `REPLICA` and stack them the same way.

## 3. Proteins with several domains: whole vs. parts

### The idea

A **domain** is a part of a protein that folds into its own compact unit. 1k5n_A is the heavy chain of the human immune protein HLA-B\*27:09 (an MHC class I molecule, 276 residues). It has two parts: a **peptide-binding platform** (the α1 and α2 regions, roughly the first 180 residues, which together hold a short peptide in a groove) and an **α3 domain** (roughly the last 95 residues, an immunoglobulin-like domain). The platform itself has two halves (α1 and α2), so it can also be split into three parts. 16pk_A has two domains (visible as two blocks in its contact map).

Note: the ATLAS entry contains only chain A. In the crystal structure, the heavy chain is bound to a second protein (β2-microglobulin) and a peptide, which are missing here, so the α3 domain and the groove may move more than they would in the full complex.

In a multi-domain protein, two kinds of motion mix:
- **inside a domain**: local changes, like the 1a62_A transition,
- **between domains**: the domains moving relative to each other, e.g. a hinge opening and closing.

Projecting the whole protein mixes both. Projecting each part separately shows which part is responsible for a pattern.

### The parts

Every residue pair (column of the contact table) belongs to exactly one part:
- **intra-domain contacts of domain k**: both residues in domain k,
- **inter-domain contacts**: the residues are in different domains.

Each part is projected on its own, with the same windows and PCA as before.

### Domain boundaries

Two options:
- **Manual:** type the boundaries in `MANUAL_DOMAINS` (0-based positions in the CA list), e.g. from CATH. Check how ATLAS numbers the residues first: the ATLAS chain can be shorter than the PDB chain.
- **Automatic (default):** spectral clustering on the average contact map splits the residues into `n_domains` groups that have many contacts inside and few between. Isolated residues are smoothed out. Always compare the result with the contact map and, if possible, with CATH.

In [ ]:
from sklearn.cluster import SpectralClustering
from scipy.ndimage import median_filter

DOMAIN_SETUPS = {
    "1k5n_A": {"n_domains": 2, "manual": None},   # try 3 as well: alpha1 / alpha2 / alpha3
    "16pk_A": {"n_domains": 2, "manual": None},
}
# Manual example (0-based CA positions, end exclusive; a domain can have several segments):
# Approximate MHC class I boundaries, assuming the ATLAS chain starts at residue 1 (check the PDB file first):
# DOMAIN_SETUPS["1k5n_A"]["manual"] = {"platform (alpha1+alpha2)": [(0, 180)], "alpha3": [(180, 276)]}
# DOMAIN_SETUPS["1k5n_A"]["manual"] = {"alpha1": [(0, 90)], "alpha2": [(90, 180)], "alpha3": [(180, 276)]}


def contact_frequency(coords, threshold=CONTACT_THRESHOLD):
    return squareform(contact_table(coords, threshold).mean(axis=0))


def assign_domains(coords, n_domains, manual=None, smooth=9):
    """Domain label (0..n-1) per residue and the domain names."""
    N = coords.shape[1]
    if manual:
        labels = np.full(N, -1)
        for k, segments in enumerate(manual.values()):
            for start, end in segments:
                labels[start:end] = k
        if (labels < 0).any():
            print(f"warning: {(labels < 0).sum()} residues not in any domain; assigned to the nearest one")
            for r in np.flatnonzero(labels < 0):
                labels[r] = labels[np.flatnonzero(labels >= 0)[np.argmin(np.abs(np.flatnonzero(labels >= 0) - r))]]
        return labels, list(manual)
    freq = contact_frequency(coords)
    labels = SpectralClustering(n_clusters=n_domains, affinity="precomputed",
                                random_state=RANDOM_STATE).fit_predict(freq + 1e-6)
    labels = median_filter(labels, size=smooth, mode="nearest")        # remove isolated residues
    order = {old: new for new, old in enumerate(dict.fromkeys(labels))}  # number domains along the chain
    labels = np.array([order[l] for l in labels])
    return labels, [f"domain {k}" for k in range(labels.max() + 1)]


def segments(labels, k):
    idx = np.flatnonzero(labels == k)
    breaks = np.flatnonzero(np.diff(idx) > 1)
    starts, ends = np.r_[idx[0], idx[breaks + 1]], np.r_[idx[breaks], idx[-1]]
    return ", ".join(f"{s}-{e}" for s, e in zip(starts, ends))


domains = {}
for p, setup in DOMAIN_SETUPS.items():
    if p not in data:
        continue
    labels, names = assign_domains(data[p]["coords"], setup["n_domains"], setup["manual"])
    domains[p] = {"labels": labels, "names": names}
    print(p)
    for k, name in enumerate(names):
        print(f"  {name}: residues {segments(labels, k)} ({(labels == k).sum()} residues)")

### Check the domains on the contact map

The average contact map with the domain of each residue as a coloured strip along both axes, and dashed lines at the boundaries.
Good domains look like dense square blocks on the diagonal with few contacts between them.

In [ ]:
fig, axes = plt.subplots(1, len(domains), figsize=(7 * len(domains), 6), squeeze=False)
for ax, (p, d) in zip(axes[0], domains.items()):
    labels = d["labels"]
    N = len(labels)
    ax.imshow(contact_frequency(data[p]["coords"]), origin="lower", cmap="Greys", vmin=0, vmax=1,
              interpolation="nearest")
    colours = plt.cm.tab10(labels)
    ax.scatter(np.arange(N), np.full(N, -4), c=colours, s=6, marker="s", clip_on=False)
    ax.scatter(np.full(N, -4), np.arange(N), c=colours, s=6, marker="s", clip_on=False)
    for b in np.flatnonzero(np.diff(labels)) + 0.5:
        ax.axvline(b, ls="--", lw=0.6, c="red"); ax.axhline(b, ls="--", lw=0.6, c="red")
    ax.set_xlim(-6, N); ax.set_ylim(-6, N)
    ax.set_title(f"{p}: average contact map and domains")
    ax.set_xlabel("Residue index"); ax.set_ylabel("Residue index")
plt.tight_layout()
plt.show()

### Projections: whole protein vs. each domain vs. between domains

One column per part. Top: PCA projection coloured by time. Bottom: PC1 over time.
The title shows how many contacts change in that part.

**How to read it**

- A transition visible in the whole protein **and** in one domain's own contacts happens inside that domain.
- A transition visible in the whole protein **and** in the inter-domain contacts, but not inside the domains, is a motion of the domains relative to each other (e.g. a hinge).
- A pattern that only shows up in one part (not in the whole) was drowned out by the rest of the protein: projecting parts separately reveals it.

In [ ]:
def part_masks(labels, names):
    N = len(labels)
    I, J = np.triu_indices(N, k=1)
    li, lj = labels[I], labels[J]
    masks = {"whole protein": np.ones(len(I), bool)}
    for k, name in enumerate(names):
        masks[f"inside {name}"] = (li == k) & (lj == k)
    masks["between domains"] = li != lj
    return masks


def part_projections(p, labels, names):
    C = contact_table(data[p]["coords"])
    changing = varying_columns(C)
    t = window_mean(data[p]["time_ns"])
    results = {}
    for part, mask in part_masks(labels, names).items():
        X = window_mean(C[:, mask & changing].astype(np.float32))
        keep = varying_columns(X)
        if keep.sum() < 2:
            print(f"{p}, {part}: fewer than 2 changing contacts, skipped")
            continue
        Y = PCA(n_components=2).fit_transform(X[:, keep])
        results[part] = {"Y": Y, "n_changing": int(keep.sum())}
    return results, t


for p, d in domains.items():
    results, t = part_projections(p, d["labels"], d["names"])
    fig, axes = plt.subplots(2, len(results), figsize=(5 * len(results), 8.5), squeeze=False)
    for col, (part, r) in enumerate(results.items()):
        plot_projection(axes[0, col], r["Y"], t, f"{part}\n({r['n_changing']} changing contacts)", "Time (ns)")
        axes[1, col].plot(t, r["Y"][:, 0], lw=0.8)
        axes[1, col].set_xlabel("Time (ns)"); axes[1, col].set_ylabel("PC1")
    fig.suptitle(f"{p}: whole protein vs. parts", y=1.0)
    plt.tight_layout()
    plt.show()

### Domain-level summary over time

Two simple readouts of how the domains move relative to each other:

- **Contacts between each pair of domains** over time: drops mean the domains move apart at that interface.
- **Distance between domain centres** (average CA position of each domain): an opening hinge shows up as a growing distance.

These help interpret the "between domains" projection above: a jump in its PC1 should coincide with a change here.

In [ ]:
for p, d in domains.items():
    labels, names = d["labels"], d["names"]
    coords, t_raw = data[p]["coords"], data[p]["time_ns"]
    K = labels.max() + 1
    pairs = [(a, b) for a in range(K) for b in range(a + 1, K)]

    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    t = window_mean(t_raw)
    for a, b in pairs:
        ia, ib = np.flatnonzero(labels == a), np.flatnonzero(labels == b)
        n_contacts = np.array([(np.linalg.norm(f[ia][:, None] - f[ib][None], axis=-1) < CONTACT_THRESHOLD).sum()
                               for f in coords])
        centre_dist = np.linalg.norm(coords[:, ia].mean(1) - coords[:, ib].mean(1), axis=-1)
        axes[0].plot(t, window_mean(n_contacts), lw=0.8, label=f"{names[a]} - {names[b]}")
        axes[1].plot(t, window_mean(centre_dist), lw=0.8, label=f"{names[a]} - {names[b]}")
    axes[0].set_ylabel("Contacts between domains"); axes[1].set_ylabel("Distance between domain centres (Å)")
    for ax in axes:
        ax.set_xlabel("Time (ns)"); ax.legend(fontsize=8)
    fig.suptitle(p)
    plt.tight_layout()
    plt.show()

### Centrality per domain (optional combination of 1 and 3)

Betweenness is a natural fit for multi-domain proteins: residues at the domain interfaces carry the shortest paths between domains.
This plot shows each residue's betweenness over time with the domain boundaries marked; a hinge motion should show up as a change in the interface residues.

In [ ]:
for p, d in domains.items():
    labels = d["labels"]
    X = window_mean(cent[p]["betweenness"])
    t = window_mean(data[p]["time_ns"])
    N = X.shape[1]
    fig, ax = plt.subplots(figsize=(13, 5))
    im = ax.imshow(X.T, aspect="auto", origin="lower", cmap="magma",
                   extent=[t[0], t[-1], -0.5, N - 0.5], interpolation="nearest")
    for b in np.flatnonzero(np.diff(labels)) + 0.5:
        ax.axhline(b, ls="--", lw=0.8, c="cyan")
    ax.set_xlabel("Time (ns)"); ax.set_ylabel("Residue index")
    ax.set_title(f"{p}: betweenness per residue (dashed = domain boundaries)")
    plt.colorbar(im, ax=ax, label="betweenness")
    plt.tight_layout()
    plt.show()

## 4. Feedback checks

Three questions from the meeting:

1. **t-SNE looks like spaghetti.** Does a different perplexity help?
2. **There are no repetitions** (no places where early and late points overlap). Did the filtering remove them, or does the data not have any?
3. **The mdCATH shared projection looks odd.** What is going on?

These cells reuse the variables from sections 1–3, so run them after the rest of the notebook (or copy them to the end of a notebook that has already run).

### 4.1 t-SNE perplexity

**Why the t-SNE plots look like spaghetti.** t-SNE keeps each point next to its nearest neighbours. With overlapping windows, two consecutive windows share 16 of their 20 frames, so a window's nearest neighbours are almost always the windows just before and after it.
t-SNE then draws one long chain in time order: a thread, not clusters.

**Perplexity** is roughly the number of neighbours t-SNE pays attention to for each point (default 30). A larger perplexity makes each point consider more neighbours, including windows from other times in the same state, so states should come out as blobs instead of threads.
It must stay below about a third of the number of points.

The grid below tries perplexity 30, 100 and 300 on two inputs:
- the **1 ns windows with 0.8 overlap** (as in section 1, about 496 points),
- **single frames** without windows (2,001 points). Consecutive frames still look alike, but less so than overlapping windows.

**What to look for:** compact blobs that contain several time colours (a state the protein stays in or returns to), instead of a thread whose colour changes smoothly along its length.

In [ ]:
PERPLEXITIES = [30, 100, 300]
TSNE_REPRESENTATION = "edge vector"      # or "betweenness", "degree", ...
p = PROTEIN

if TSNE_REPRESENTATION == "edge vector":
    single = changing_contacts(data[p]["coords"])
else:
    single = cent[p][TSNE_REPRESENTATION]
tsne_inputs = {
    "1 ns windows, 0.8 overlap": (window_mean(single), window_mean(data[p]["time_ns"])),
    "single frames": (single, data[p]["time_ns"]),
}

fig, axes = plt.subplots(len(tsne_inputs), len(PERPLEXITIES), figsize=(6 * len(PERPLEXITIES), 5 * len(tsne_inputs)))
for row, (name, (X, t)) in zip(axes, tsne_inputs.items()):
    X = X[:, varying_columns(X)]
    for ax, perp in zip(row, PERPLEXITIES):
        perp_used = min(perp, (len(X) - 1) // 3)
        Y = TSNE(n_components=2, perplexity=perp_used, init="pca", random_state=RANDOM_STATE).fit_transform(X)
        plot_projection(ax, Y, t, f"{name}: perplexity {perp_used}", "Time (ns)", connect=(name != "single frames"))
fig.suptitle(f"{p}, {TSNE_REPRESENTATION}: t-SNE with different perplexities", y=1.0)
plt.tight_layout()
plt.show()

### 4.2 Are there repetitions?

**The filtering cannot remove them.** Dropping constant columns removes residue pairs (or residues) that never change; it never removes frames.
Windowing averages neighbouring frames, but if the protein returned to an earlier state, the windows from that later time would still look like the earlier ones.
So if no returns show up, the data has none, or the 2D projection hides them.

Two checks, both in the **full feature space** (all contacts or all residues), not in the 2D projection, so the projection cannot hide anything:

1. **Identical snapshots.** How many frames have exactly the same contact pattern as another frame? This is what the paper's "same points" means: in the school data, every night snapshot was identical (no contacts at all). With 400+ flickering contacts, exact copies are unlikely in MD.
2. **Recurrence plot.** The distance between every pair of windows, drawn as an image with time on both axes (like the RMSD matrix). How to read it:
    - the **diagonal** is each window compared with itself (distance 0),
    - a **dark square block on the diagonal** = a stable state: all windows in that period resemble each other,
    - a **dark block away from the diagonal**, at (early time, late time) = a **return**: the late windows resemble the early ones,
    - if everything far from the diagonal is bright, the protein never comes back.

The right panel marks only the closest 10% of window pairs (black), which makes returns easier to spot.

In [ ]:
print("Identical contact snapshots (single frames, changing contacts only):")
for p in data:
    C = contact_table(data[p]["coords"])
    C = C[:, varying_columns(C)]
    n_unique = len(np.unique(np.packbits(C, axis=1), axis=0))
    print(f"  {p}: {len(C)} frames, {n_unique} different contact patterns, {len(C) - n_unique} exact repeats")

In [ ]:
RECURRENCE_REPRESENTATIONS = ["edge vector", "betweenness"]
CLOSEST_FRACTION = 0.10

for p in data:
    reps = representations(p)
    t = window_mean(data[p]["time_ns"])
    fig, axes = plt.subplots(len(RECURRENCE_REPRESENTATIONS), 2, figsize=(13, 5.5 * len(RECURRENCE_REPRESENTATIONS)),
                             squeeze=False)
    for row, rep in zip(axes, RECURRENCE_REPRESENTATIONS):
        X = reps[rep]
        X = X[:, varying_columns(X)]
        D = squareform(pdist(X))
        eps = np.quantile(D[np.triu_indices(len(D), k=1)], CLOSEST_FRACTION)
        ext = [t[0], t[-1], t[0], t[-1]]
        im = row[0].imshow(D, origin="lower", cmap="viridis", extent=ext)
        row[0].set_title(f"{p}, {rep}: distance between windows")
        plt.colorbar(im, ax=row[0], label="distance")
        row[1].imshow(D < eps, origin="lower", cmap="Greys", extent=ext, interpolation="nearest")
        row[1].set_title(f"{p}, {rep}: closest {CLOSEST_FRACTION:.0%} of pairs (black)")
        for ax in row:
            ax.set_xlabel("Time (ns)"); ax.set_ylabel("Time (ns)")
    plt.tight_layout()
    plt.show()

**Reading tips.** In 100 ns, ATLAS proteins mostly drift or switch once (1a62_A), so expect blocks along the diagonal and few off-diagonal blocks.
Earlier results suggest where returns might appear: domain 0 of 16pk_A came back to its start after ~80 ns (section 3), and 1ab1_A from the first notebook looked like one state visited throughout.
To test a single domain, replace `reps[rep]` by the windowed contacts of that domain (see `part_masks` in section 3).

### 4.3 mdCATH: why the shared projection looks odd

Two things in the section 2 plot are side effects of PCA, not features of the protein:

1. **The folded runs are squeezed into a thin streak.** PCA picks the directions with the most variation in the *whole* table. The 450 K run unfolds and changes hundreds of contacts, so it supplies almost all the variation.
   PC1 and PC2 end up describing the unfolding, and the differences between 320, 379 and 413 K (all folded) barely register.
2. **The 450 K run traces an arch.** When data changes gradually in one direction (here: steadily losing native contacts), PCA often draws a curve: PC1 follows the change and PC2 bends into a U shape: the middle of the change sits at one extreme of PC2, both ends at the other (the U can point up or down, since the sign of PC2 is arbitrary).
   This is a known artifact, called the **arch or horseshoe effect**. PC2 then adds no new information; it is just a curved function of PC1. So the "path up and around" should not be read as the protein visiting a separate region.

The middle plot (coloured by time) adds confusion: it colours four different runs on one time scale, so purple points from different temperatures look alike.

The cells below check and fix each point:
- **Variance per run:** how much of the total variation each run contributes.
- **Small multiples:** one panel per temperature in the same shared space, with the other runs in grey.
- **Arch check:** PC1 and PC2 against the fraction of native contacts Q. If PC1 follows Q and PC2 forms a U shape against Q, the arch is the horseshoe artifact.
- **PCA fitted on the folded runs only** (320–413 K): shows how the folded runs differ, and where 450 K leaves them.

In [ ]:
FOLDED_TEMPS = [T for T in runs if T < 450]

# Contact tables and windows per run (full residue-pair table, so all runs share the same columns)
mC, mW, mt = {}, {}, {}
for T, r in runs.items():
    mC[T] = contact_table(r["coords"])
    mW[T] = window_mean(mC[T], MDCATH_WIDTH, mstep)
    mt[T] = window_mean(r["time_ns"], MDCATH_WIDTH, mstep)
keep_cols = varying_columns(np.vstack([mC[T] for T in runs]))
for T in runs:
    mW[T] = mW[T][:, keep_cols].astype(np.float32)
X_all = np.vstack([mW[T] for T in runs])
run_of = np.concatenate([np.full(len(mW[T]), T) for T in runs])

pca_all = PCA(n_components=2).fit(X_all)
Y_all = pca_all.transform(X_all)
print(f"Shared PCA: PC1 {pca_all.explained_variance_ratio_[0]:.0%}, PC2 {pca_all.explained_variance_ratio_[1]:.0%}")

centre = X_all.mean(axis=0)
total = ((X_all - centre) ** 2).sum()
print("Share of the total variation contributed by each run:")
for T in runs:
    share = ((X_all[run_of == T] - centre) ** 2).sum() / total
    print(f"  {T} K: {share:.0%}  ({(run_of == T).sum()} windows)")

In [ ]:
# Small multiples: each temperature in the shared space, the other runs in grey
lims = (Y_all[:, 0].min(), Y_all[:, 0].max()), (Y_all[:, 1].min(), Y_all[:, 1].max())
fig, axes = plt.subplots(1, len(runs), figsize=(5 * len(runs), 4.5))
for ax, T in zip(axes, runs):
    sel = run_of == T
    ax.scatter(Y_all[~sel, 0], Y_all[~sel, 1], s=3, c="lightgray")
    plot_projection(ax, Y_all[sel], mt[T], f"{T} K", "Time (ns)")
    ax.set_xlim(*lims[0]); ax.set_ylim(*lims[1])
fig.suptitle(f"{MDCATH_DOMAIN}: shared PCA, one panel per temperature", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Arch check: PC1 and PC2 against the fraction of native contacts Q
N_res = runs[FOLDED_TEMPS[0]]["coords"].shape[1]
I_all, J_all = np.triu_indices(N_res, k=1)
native = (mC[min(runs)].mean(axis=0) >= 0.9) & ((J_all - I_all) >= 3)     # contacts present in >= 90% of the coolest run
print(f"{native.sum()} native contacts (from the {min(runs)} K run)")
Q_all = np.concatenate([window_mean(mC[T][:, native].mean(axis=1), MDCATH_WIDTH, mstep) for T in runs])

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for T in runs:
    sel = run_of == T
    axes[0].plot(Y_all[sel, 0], Y_all[sel, 1], lw=0.5, alpha=0.7)
    axes[0].scatter(Y_all[sel, 0], Y_all[sel, 1], s=4, label=f"{T} K")
    axes[1].scatter(Q_all[sel], Y_all[sel, 0], s=4, label=f"{T} K")
    axes[2].scatter(Q_all[sel], Y_all[sel, 1], s=4, label=f"{T} K")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2"); axes[0].set_title("Shared PCA")
axes[1].set_xlabel("Fraction of native contacts Q"); axes[1].set_ylabel("PC1"); axes[1].set_title("PC1 vs. Q")
axes[2].set_xlabel("Fraction of native contacts Q"); axes[2].set_ylabel("PC2"); axes[2].set_title("PC2 vs. Q")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"Correlation of PC1 with Q: {np.corrcoef(Q_all, Y_all[:, 0])[0, 1]:+.2f}")

**How to read the arch check**

- **PC1 vs. Q** close to a straight line (correlation near ±1): PC1 is essentially "how folded the protein is".
- **PC2 vs. Q** shaped like a U, either way up (the middle values of Q at one extreme of PC2, both ends at the other): PC2 is the arch artifact, not a second independent motion.
- If both hold, the shared projection really shows one thing, the loss of native contacts, and the time-vs-PC1 plot from section 2 is the honest summary.

In [ ]:
# PCA fitted on the folded runs only; every run projected into that space
X_fold = np.vstack([mW[T] for T in FOLDED_TEMPS])
pca_fold = PCA(n_components=2).fit(X_fold)
print(f"Folded-only PCA: PC1 {pca_fold.explained_variance_ratio_[0]:.0%}, PC2 {pca_fold.explained_variance_ratio_[1]:.0%}")

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for T in runs:
    Yt = pca_fold.transform(mW[T])
    if T in FOLDED_TEMPS:
        axes[0].plot(Yt[:, 0], Yt[:, 1], lw=0.4, alpha=0.7)
        axes[0].scatter(Yt[:, 0], Yt[:, 1], s=4, label=f"{T} K")
    axes[1].scatter(Yt[:, 0], Yt[:, 1], s=4, label=f"{T} K")
    axes[2].plot(mt[T], Yt[:, 0], lw=0.8, label=f"{T} K")
axes[0].set_title("Folded runs only (PCA fitted on these)")
axes[1].set_title("All runs projected into the folded PCA")
axes[2].set_title("PC1 (folded PCA) over time"); axes[2].set_xlabel("Time (ns)"); axes[2].set_ylabel("PC1")
for ax in axes[:2]:
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**What to look for**

- **Left:** with the 450 K run out of the fit, do 320, 379 and 413 K separate, or overlap? Overlap means the folded state looks the same at these temperatures; separation shows which contacts the heat loosens first.
- **Middle:** where the 450 K windows leave the folded cloud, and in which direction. That direction is the first change on the way to unfolding.
- **Right:** compare with the PC1 plot in section 2. The 413 K dips (around 30 and 250 ns) and the 379 K drift at the end should be easier to see here.
- 379 K only has 130 frames; another replica (`REPLICA` in section 2) gives a full 500 ns run.

## Observations

*(Fill in after running.)*

1. **Centrality vectors:** which centrality shows the 1a62_A transition most clearly? Which residues have the largest PC1 loadings, and do they match residues 38–39 and 100–111?
2. **mdCATH:** do high-temperature runs leave the folded region? Along one path or several?
3. **Domains:** for 1k5n_A and 16pk_A, is the main pattern inside a domain or between domains? Does projecting a single domain reveal something the whole-protein projection hides?